# Training-only C3/C4 mu-power analysis

This notebook describes 8-13 Hz power during left/right trials in six training
runs from three participants. P1 PRE is the worked example; the same fixed
calculation is then applied to every training run. No classifier is fitted
and no test recording is loaded.

The analysis and notebook were prepared with AI assistance and reviewed for
this contribution. The measurements are descriptive; they do not establish
artifact-free imagery, source localisation or rehabilitation benefit.


## Context & Methods

**Concrete example first:** a trial with task power 20 and baseline power 10 has
a ratio of 2, giving `10 × log10(2) = +3.01 dB`. A ratio of 0.5 gives −3.01 dB.
Positive means **more power**, negative means **less power**. Neither means
"correct answer" or "successful rehabilitation".

Mu here means the chosen 8–13 Hz frequency band; activity in this band is not
automatically a clean motor-cortex signal. C3/C4 are **scalp electrodes**, on the
left/right sides of the montage. A rhythm's power measures its strength, rather
than how fast the rhythm oscillates.

Input → separate trial/baseline windows → estimate each window's spectrum → add
8–13 Hz power → calculate the same-trial task/baseline ratio → show every trial.

### Key Assumptions

- The supplied `montage.png` maps C3 to EEG column 5 and C4 to column 9. The
  accompanying paper uses a different montage; its order is not substituted.
- `+1` means left, `−1` right. Each label lasts eight seconds; 2,048 labelled
  samples are **one trial**, not 2,048 independent observations.
- Times are relative to labelled-block onset: attention at 0 s, cue at 2 s,
  relax at 8 s. Baseline is [−1, 0), full task [2, 8), early [2, 3.5).
- The diagram puts feedback around 3.5 s. Its actual per-trial onset is absent.
  Full task includes feedback; early is a candidate timing control, not proof
  of feedback-free imagery.
- Welch: one-second Hann segments, 50% overlap, per-segment mean removal,
  one-Hz frequency grid; trapezoidal integration from 8 through 13 Hz. No
  continuous noncausal filtering, rereferencing or artifact rejection.
- The baseline has one segment; early two; full task eleven. Their precision
  differs. A low/noisy baseline can inflate the ratio. dB removes an unknown
  fixed gain, not reference effects or artifacts.

Estimator source: [SciPy Welch documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.welch.html).
All parameters are explicit; installed versions are recorded in the provenance.


### Setup

Install the repository with its notebook dependencies: `python -m pip install -e '.[notebook]'`.
Use the organiser's local files in `data/stroke-rehab`, or set `STROKE_REHAB_DATA`
(or `G27_DATA_DIR`) to your extracted data folder. Run every cell from top to
bottom. Raw data are not included or downloaded by this notebook.


In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np
from IPython.display import display, Image, Markdown

starting_folder = Path.cwd().resolve()
project_root = next(folder for folder in [starting_folder, *starting_folder.parents]
                    if (folder / 'analysis' / 'mu_power' / 'mu_pass.py').is_file())
code_folder = project_root / 'analysis' / 'mu_power'
sys.path.insert(0, str(code_folder))
from mu_pass import (DEFAULT_DATA, DEFAULT_OUTPUT, CHANNELS, read_training,
                     get_epochs, mu_power, analyze_training, plot_p1, run_pass,
                     verify_estimator)
data_folder = Path(os.environ.get('STROKE_REHAB_DATA',
                   os.environ.get('G27_DATA_DIR', str(DEFAULT_DATA))))
output_folder = DEFAULT_OUTPUT
output_folder.mkdir(parents=True, exist_ok=True)
os.environ['MPLCONFIGDIR'] = str(output_folder / '.matplotlib-cache')
print('Training data folder available:', data_folder.is_dir())
print('Output folder: results/mu_power')
print('C3/C4 zero-based column indices:', CHANNELS)


## Data

### 1. Read P1 PRE training and verify trial structure

The loader checks shape, sampling rate, labels and every baseline. It stops on
invalid data rather than quietly dropping trials. The file contains 80 trials,
40 for each hand, from one training run of one participant.


In [ ]:
recording = read_training(data_folder / 'P1_pre_training.mat')
print('EEG shape (samples, channels):', recording['eeg'].shape)
print('Sampling rate:', recording['fs'], 'Hz')
print('Trials:', len(recording['onsets']))
print('Left / right:', np.sum(recording['labels'] == 1), '/', np.sum(recording['labels'] == -1))
print('First onset sample:', recording['onsets'][0], '(zero based)')
print('First label:', recording['labels'][0], '=> right')
print('Checks:', recording['checks'])


### 2. Cut out baseline and task samples

Python's end index is excluded. At 256 Hz, baseline has 256 samples, early
has 384 and full task 1,536. The cue is 512 samples after block onset.
Epoch means one cut-out segment of the continuous recording.


In [ ]:
baseline_epochs = get_epochs(recording, -1, 0)
full_task_epochs = get_epochs(recording, 2, 8)
early_epochs = get_epochs(recording, 2, 3.5)
for name, epochs in [('baseline', baseline_epochs), ('early', early_epochs), ('full task', full_task_epochs)]:
    print(name, epochs.shape, '(trials, C3/C4, samples)')
assert baseline_epochs.shape == (80, 2, 256)
assert full_task_epochs.shape == (80, 2, 1536)


## Results

### 3. Follow the first C3 trial through the calculation

Power is in the square of the recording's original units. Physical amplitude
units are unverified, so we do not label it µV². Dividing task by baseline
cancels a constant recording scale. This does not remove contamination.


In [ ]:
baseline_power = mu_power(baseline_epochs)
full_task_power = mu_power(full_task_epochs)
change_db = 10 * np.log10(full_task_power / baseline_power)
print('First trial / C3 / right hand')
print(f'Baseline mu power: {baseline_power[0, 0]:.6f} original units squared')
print(f'Task mu power:     {full_task_power[0, 0]:.6f} original units squared')
print(f'Task / baseline:  {full_task_power[0, 0] / baseline_power[0, 0]:.6f}')
print(f'Change:           {change_db[0, 0]:+.6f} dB')
assert np.isfinite(change_db).all()


### 4. Show all P1 trials, not a selected example

Each point is one trial at one electrode. Black bars show medians; the dashed
line is the trial baseline. The two panels use the same vertical scale and
retain the full range. Jitter only separates dots horizontally.


In [ ]:
p1 = analyze_training(data_folder / 'P1_pre_training.mat')
np.testing.assert_allclose(p1['windows']['full_task']['db'], change_db)
p1_paths = plot_p1(p1, output_folder)
display(Image(filename=p1_paths['full_task']['png']))
table = '| Electrode | Left median | Right median | Left IQR | Right IQR |\n|---|---:|---:|---:|---:|\n'
for channel in CHANNELS:
    left = next(row for row in p1['summaries'] if row['window'] == 'full_task' and row['channel'] == channel and row['hand'] == 'left')
    right = next(row for row in p1['summaries'] if row['window'] == 'full_task' and row['channel'] == channel and row['hand'] == 'right')
    table += f"| {channel} | {left['median_db']:+.2f} dB | {right['median_db']:+.2f} dB | {left['q25_db']:+.2f} to {left['q75_db']:+.2f} | {right['q25_db']:+.2f} to {right['q75_db']:+.2f} |\n"
display(Markdown(table))


The four medians are positive. Left trials are higher than right trials at
**both** electrodes; that is not a simple opposite-side decrease pattern.
The interquartile range (IQR) spans the middle half of the trials. These ranges
overlap. We have not tested a population-level effect or explained its cause.

One participant's trial count is not a sample of 80 independent patients.
This summary averages over time and uses only two electrodes and one band;
the team's classifier uses 16 channels, three bands and learned spatial
combinations. Simple plot overlap does not determine its accuracy.


### 5. Fixed earlier window

This uses [2, 3.5) s without consulting later samples. That addresses the
specific backward-filtering concern, but uncertainty in feedback timing,
cue responses, artifacts and shorter spectral averaging remain.


In [ ]:
display(Image(filename=p1_paths['early']['png']))
for row in p1['summaries']:
    if row['window'] == 'early':
        print(row['channel'], row['hand'], f"median {row['median_db']:+.2f} dB")


### 6. Same calculation for the other training runs

This extension keeps the same electrodes, band, baseline and task window.
PRE/POST label first/last rehabilitation sessions; each has a different run.
The figure compares descriptions, not a controlled treatment effect.


In [ ]:
bundle = run_pass(data_folder, output_folder)
display(Image(filename=bundle['paths']['extension']['png']))
print('Training runs:', len(bundle['results']))
print('Trials:', sum(len(result['recording']['onsets']) for result in bundle['results']))
print('Trial x channel rows:', sum(len(result['rows']) for result in bundle['results']))
print('Participants: 3 (not 480)')


Direction and size vary across runs. For example, P3 PRE has C3 medians
+1.40/−0.51 dB (left/right), but C4 −0.70/+0.98 dB. P2 PRE's C3 medians are
almost equal (+1.61/+1.62 dB). This is a useful reason to report individuals
and avoid telling one universal rehabilitation story.


### 7. Check the calculations and separate reproduction from new analysis

A synthetic 10-Hz rhythm should preserve power when given a constant offset,
and doubling its amplitude should quadruple power (+6.02 dB). The export also
checks all baseline windows, a separately computed band integral, CSV readback
and unchanged input hashes. Technical correctness is not scientific validation.


In [ ]:
print('Synthetic estimator checks:', verify_estimator())
for source in bundle['provenance']['inputs']:
    assert all(source['checks'].values())
    print(source['file'], 'PASS', source['sha256'][:12])
assert bundle['provenance']['csv_roundtrip_checks']
print('CSV readback: PASS')
print('No classifier or test recording was used for the mu-power calculations.')


## Interpretation

P1 PRE full-task medians are C3: left +1.34 / right +0.36 dB, and
C4: left +1.40 / right +0.46 dB. Distributions overlap; positive values mean
more band power than the same-trial baseline. Other runs differ.

The [2, 8) s window includes documented feedback. The [2, 3.5) s window uses
only samples in that interval, but actual feedback timestamps are absent.
Baseline and task windows have different spectral averaging precision.
No artifacts are removed, and the original reference is retained. Scalp
measurements do not establish an individual cortical source or treatment effect.

The fixed settings, input hashes, estimator checks and CSV readback are saved
in `results/mu_power/provenance_and_checks.json`. See
[the analysis notes](../analysis/mu_power/README.md) for reproduction and limits.
